In [ ]:
pip install seaborn  dash dash-bootstrap-components flask-ngrok cryptography tenseal psutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 98.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import os
import random
import shutil
import pandas as pd
from typing import Dict, Set
from scipy.stats import dirichlet

# =====================================================
# Reproducibility
# =====================================================

np.random.seed(42)
random.seed(42)

# =====================================================
# Convert CSV chess dataset → FCA formal context
# =====================================================

def process_chess_csv_dataset(input_file):
    """
    Convert chess CSV dataset into FCA formal context.

    Each row = one object.
    Each categorical value becomes one FCA attribute.
    Example:
        winner=white
        victory_status=mate
        opening_eco=C20
    """

    df = pd.read_csv(input_file)

    # Optional: remove columns that are IDs or too unique
    columns_to_drop = [
        "id",
        "created_at",
        "last_move_at",
        "white_id",
        "black_id",
        "moves"
    ]

    df = df.drop(columns=[c for c in columns_to_drop if c in df.columns])

    formal_context_dict = {}
    attribute_mapping = {}
    next_attr_id = 1

    for obj_idx, row in enumerate(df.itertuples(index=False), start=1):

        attributes = set()

        for col_name, value in zip(df.columns, row):

            if pd.isna(value):
                continue

            attr_name = f"{col_name}={value}"

            if attr_name not in attribute_mapping:
                attribute_mapping[attr_name] = next_attr_id
                next_attr_id += 1

            attributes.add(attribute_mapping[attr_name])

        formal_context_dict[obj_idx] = attributes

    return formal_context_dict, attribute_mapping


# =====================================================
# Save formal context
# =====================================================

def save_formal_context(formal_context_dict, filename):

    with open(filename, "w", encoding="utf-8") as file:

        for obj_idx, attributes in formal_context_dict.items():

            file.write(f"{obj_idx}: {sorted(attributes)}\n")

    print(f"Formal context saved: {filename}")


# =====================================================
# Save attribute mapping
# =====================================================

def save_attribute_mapping(attribute_mapping, filename):

    with open(filename, "w", encoding="utf-8") as file:

        for attr_name, attr_id in sorted(attribute_mapping.items(), key=lambda x: x[1]):

            file.write(f"{attr_id}: {attr_name}\n")

    print(f"Attribute mapping saved: {filename}")


# =====================================================
# Federated Partitioning
# =====================================================

def split_dataset_into_providers_fca(
        formal_context_dict: Dict[int, Set[int]],
        num_providers: int,
        splitting_type="IID",
        skew_factor=1.0,
        attribute_distribution_skew=False,
        quantity_skew=False,
        max_objects_per_provider=None):

    object_ids = np.array(list(formal_context_dict.keys()))

    all_attributes = list(
        {attr for attrs in formal_context_dict.values() for attr in attrs}
    )

    attribute_index = {attr: i for i, attr in enumerate(all_attributes)}

    if splitting_type == "IID":

        shuffled_indices = np.random.permutation(object_ids)
        provider_indices = np.array_split(shuffled_indices, num_providers)

    elif splitting_type == "Non-IID":

        provider_indices = [[] for _ in range(num_providers)]

        if attribute_distribution_skew:

            attribute_proportions = dirichlet.rvs(
                [skew_factor] * num_providers,
                size=len(all_attributes)
            )

            for obj_idx, attributes in formal_context_dict.items():

                provider_weights = np.zeros(num_providers)

                for attr in attributes:
                    attr_idx = attribute_index[attr]
                    provider_weights += attribute_proportions[attr_idx]

                total_weight = provider_weights.sum()

                if total_weight == 0:
                    provider_choice = np.random.randint(num_providers)
                else:
                    provider_weights = provider_weights / total_weight
                    provider_choice = np.random.choice(
                        range(num_providers),
                        p=provider_weights
                    )

                provider_indices[provider_choice].append(obj_idx)

        elif quantity_skew:

            proportions = dirichlet.rvs(
                [skew_factor] * num_providers,
                size=1
            )[0]

            shuffled_objects = np.random.permutation(object_ids)

            for obj_idx in shuffled_objects:

                provider_choice = np.random.choice(
                    num_providers,
                    p=proportions
                )

                provider_indices[provider_choice].append(obj_idx)

        else:
            raise ValueError(
                "For Non-IID, set attribute_distribution_skew=True or quantity_skew=True"
            )

        # Avoid empty providers
        for provider_idx, indices in enumerate(provider_indices):

            if not indices:

                fullest = max(
                    range(num_providers),
                    key=lambda x: len(provider_indices[x])
                )

                provider_indices[provider_idx].append(
                    provider_indices[fullest].pop()
                )

    else:
        raise ValueError("Unknown splitting type. Use IID or Non-IID.")

    provider_data = []

    for indices in provider_indices:

        if max_objects_per_provider is not None:
            indices = indices[:max_objects_per_provider]

        provider_dict = {
            obj_idx: formal_context_dict[obj_idx]
            for obj_idx in indices
        }

        provider_data.append(provider_dict)

    return provider_data


# =====================================================
# Save provider data
# =====================================================

def save_provider_data(dataset_name,
                       num_providers,
                       provider_idx,
                       provider_data,
                       splitting_type):

    provider_folder = os.path.join(
        dataset_name,
        str(num_providers),
        splitting_type
    )

    os.makedirs(provider_folder, exist_ok=True)

    provider_file = os.path.join(
        provider_folder,
        f"provider_{provider_idx}.txt"
    )

    with open(provider_file, "w", encoding="utf-8") as file:

        for obj_idx, attributes in provider_data.items():

            file.write(f"{obj_idx}: {sorted(attributes)}\n")

    print(f"Saved {provider_file}")


# =====================================================
# Create federated chess dataset
# =====================================================

def create_federated_chess_dataset(
        input_file,
        dataset_name="chess",
        num_providers_list=[100],
        splitting_types=["IID", "Non-IID"],
        skew_factor=0.2,
        attribute_distribution_skew=True,
        quantity_skew=False,
        max_objects_per_provider=1000):

    print(f"\nProcessing dataset: {dataset_name}")

    if not os.path.exists(input_file):
        print(f"Dataset not found: {input_file}")
        return

    formal_context, attribute_mapping = process_chess_csv_dataset(input_file)

    print(f"Number of objects: {len(formal_context)}")
    print(f"Number of attributes: {len(attribute_mapping)}")

    if os.path.exists(dataset_name):
        shutil.rmtree(dataset_name)

    os.makedirs(dataset_name)

    save_formal_context(
        formal_context,
        os.path.join(dataset_name, f"{dataset_name}.data")
    )

    save_attribute_mapping(
        attribute_mapping,
        os.path.join(dataset_name, f"{dataset_name}_attribute_mapping.txt")
    )

    for num_providers in num_providers_list:

        for splitting_type in splitting_types:

            providers = split_dataset_into_providers_fca(
                formal_context_dict=formal_context,
                num_providers=num_providers,
                splitting_type=splitting_type,
                skew_factor=skew_factor,
                attribute_distribution_skew=attribute_distribution_skew,
                quantity_skew=quantity_skew,
                max_objects_per_provider=max_objects_per_provider
            )

            for idx, pdata in enumerate(providers, start=1):

                save_provider_data(
                    dataset_name=dataset_name,
                    num_providers=num_providers,
                    provider_idx=idx,
                    provider_data=pdata,
                    splitting_type=splitting_type
                )


# =====================================================
# Run
# =====================================================

create_federated_chess_dataset(
    input_file="chess.csv",
    dataset_name="chess",
    num_providers_list=[50,100, 150, 200, 250],
    splitting_types=["IID", "Non-IID"],
    skew_factor=0.2,
    attribute_distribution_skew=True,
    quantity_skew=False,
    max_objects_per_provider=20
)


Processing dataset: chess
Number of objects: 20058
Number of attributes: 5522
Formal context saved: chess/chess.data
Attribute mapping saved: chess/chess_attribute_mapping.txt
Saved chess/50/IID/provider_1.txt
Saved chess/50/IID/provider_2.txt
Saved chess/50/IID/provider_3.txt
Saved chess/50/IID/provider_4.txt
Saved chess/50/IID/provider_5.txt
Saved chess/50/IID/provider_6.txt
Saved chess/50/IID/provider_7.txt
Saved chess/50/IID/provider_8.txt
Saved chess/50/IID/provider_9.txt
Saved chess/50/IID/provider_10.txt
Saved chess/50/IID/provider_11.txt
Saved chess/50/IID/provider_12.txt
Saved chess/50/IID/provider_13.txt
Saved chess/50/IID/provider_14.txt
Saved chess/50/IID/provider_15.txt
Saved chess/50/IID/provider_16.txt
Saved chess/50/IID/provider_17.txt
Saved chess/50/IID/provider_18.txt
Saved chess/50/IID/provider_19.txt
Saved chess/50/IID/provider_20.txt
Saved chess/50/IID/provider_21.txt
Saved chess/50/IID/provider_22.txt
Saved chess/50/IID/provider_23.txt
Saved chess/50/IID/provider

In [ ]:
import numpy as np
import os
import random
import shutil
from typing import Dict, Set
from scipy.stats import dirichlet


# =====================================================
# Reproducibility
# =====================================================

np.random.seed(42)
random.seed(42)


# =====================================================
# Convert transaction dataset (.dat) → FCA formal context
# =====================================================

def process_transaction_dataset(input_file):

    formal_context_dict = {}

    with open(input_file, "r") as file:

        for obj_idx, line in enumerate(file, start=1):

            line = line.strip()

            if not line:
                continue

            items = line.replace("\t", " ").split()

            attributes = set(map(int, items))

            formal_context_dict[obj_idx] = attributes

    return formal_context_dict


# =====================================================
# Save FCA context
# =====================================================

def save_formal_context(formal_context_dict, filename):

    with open(filename, "w") as file:

        for obj_idx, attributes in formal_context_dict.items():

            file.write(f"{obj_idx}: {sorted(attributes)}\n")

    print("Formal context saved:", filename)


# =====================================================
# Federated partitioning
# =====================================================

def split_dataset_into_providers_fca(
        formal_context_dict: Dict[int, Set[int]],
        num_providers: int,
        splitting_type="IID",
        skew_factor=1.0,
        attribute_distribution_skew=False,
        quantity_skew=False,
        max_objects_per_provider=None):


    num_objects = len(formal_context_dict)

    object_ids = np.array(list(formal_context_dict.keys()))

    all_attributes = list(
        {attr for attrs in formal_context_dict.values() for attr in attrs}
    )

    attribute_index = {attr: i for i, attr in enumerate(all_attributes)}


    # =================================================
    # IID splitting
    # =================================================

    if splitting_type == "IID":

        shuffled = np.random.permutation(object_ids)

        provider_indices = np.array_split(shuffled, num_providers)


    # =================================================
    # Non IID splitting
    # =================================================

    elif splitting_type == "Non-IID":

        provider_indices = [[] for _ in range(num_providers)]


        # -------------------------
        # Attribute skew
        # -------------------------

        if attribute_distribution_skew:

            attribute_proportions = dirichlet.rvs(
                [skew_factor] * num_providers,
                size=len(all_attributes)
            )

            for obj_idx, attributes in formal_context_dict.items():

                provider_weights = np.zeros(num_providers)

                for attr in attributes:

                    attr_idx = attribute_index[attr]

                    provider_weights += attribute_proportions[attr_idx]

                total = provider_weights.sum()

                if total == 0:
                    provider_choice = np.random.randint(num_providers)

                else:
                    provider_weights /= total

                    provider_choice = np.random.choice(
                        range(num_providers),
                        p=provider_weights
                    )

                provider_indices[provider_choice].append(obj_idx)


        # -------------------------
        # Quantity skew
        # -------------------------

        elif quantity_skew:

            proportions = dirichlet.rvs(
                [skew_factor] * num_providers,
                size=1
            )[0]

            shuffled = np.random.permutation(object_ids)

            for obj_idx in shuffled:

                provider_choice = np.random.choice(
                    num_providers,
                    p=proportions
                )

                if max_objects_per_provider and \
                        len(provider_indices[provider_choice]) >= max_objects_per_provider:

                    available = [
                        i for i in range(num_providers)
                        if len(provider_indices[i]) < max_objects_per_provider
                    ]

                    if available:
                        provider_choice = np.random.choice(available)
                    else:
                        break

                provider_indices[provider_choice].append(obj_idx)


        # -------------------------
        # Avoid empty providers
        # -------------------------

        for i, indices in enumerate(provider_indices):

            if not indices:

                fullest = max(
                    range(num_providers),
                    key=lambda x: len(provider_indices[x])
                )

                provider_indices[i].append(
                    provider_indices[fullest].pop()
                )

    else:

        raise ValueError("Unknown splitting type")


    # =================================================
    # Build provider datasets
    # =================================================

    provider_data = []

    for indices in provider_indices:

        provider_dict = {
            obj_idx: formal_context_dict[obj_idx]
            for obj_idx in indices
        }

        provider_data.append(provider_dict)

    return provider_data


# =====================================================
# Save provider data
# =====================================================

def save_provider_data(dataset_name,
                       num_providers,
                       provider_idx,
                       provider_data,
                       splitting_type):

    provider_folder = os.path.join(
        dataset_name,
        str(num_providers),
        splitting_type
    )

    os.makedirs(provider_folder, exist_ok=True)

    provider_file = os.path.join(
        provider_folder,
        f"provider_{provider_idx}.txt"
    )

    with open(provider_file, "w") as file:

        for obj_idx, attributes in provider_data.items():

            file.write(f"{obj_idx}: {sorted(attributes)}\n")

    print("Saved:", provider_file)


# =====================================================
# Create federated datasets
# =====================================================

def create_federated_datasets(
        dataset_files,
        num_providers_list,
        splitting_types,
        skew_factor=1.0,
        attribute_distribution_skew=False,
        quantity_skew=False,
        max_objects_per_provider=None):


    for dataset_name, file_path in dataset_files.items():

        print("\nProcessing dataset:", dataset_name)

        if not os.path.exists(file_path):

            print("Dataset not found:", file_path)
            continue


        formal_context = process_transaction_dataset(file_path)


        if os.path.exists(dataset_name):
            shutil.rmtree(dataset_name)

        os.makedirs(dataset_name)


        save_formal_context(
            formal_context,
            os.path.join(dataset_name, f"{dataset_name}.data")
        )


        for num_providers in num_providers_list:

            for splitting_type in splitting_types:

                providers = split_dataset_into_providers_fca(
                    formal_context,
                    num_providers,
                    splitting_type,
                    skew_factor,
                    attribute_distribution_skew,
                    quantity_skew,
                    max_objects_per_provider
                )


                for idx, pdata in enumerate(providers, start=1):

                    save_provider_data(
                        dataset_name,
                        num_providers,
                        idx,
                        pdata,
                        splitting_type
                    )


# =====================================================
# Local transaction datasets (.dat)
# =====================================================

dataset_files = {

    "mushroom": "mushroom.dat"
}

num_providers_list = [50,100, 150, 200, 250]

splitting_types = ["IID", "Non-IID"]

skew_factor = 0.2

max_objects_per_provider = 20


# =====================================================
# Run
# =====================================================

create_federated_datasets(

    dataset_files=dataset_files,

    num_providers_list=num_providers_list,

    splitting_types=splitting_types,

    skew_factor=skew_factor,

    attribute_distribution_skew=True,

    quantity_skew=True,

    max_objects_per_provider=max_objects_per_provider
)


Processing dataset: mushroom
Formal context saved: mushroom/mushroom.data
Saved: mushroom/50/IID/provider_1.txt
Saved: mushroom/50/IID/provider_2.txt
Saved: mushroom/50/IID/provider_3.txt
Saved: mushroom/50/IID/provider_4.txt
Saved: mushroom/50/IID/provider_5.txt
Saved: mushroom/50/IID/provider_6.txt
Saved: mushroom/50/IID/provider_7.txt
Saved: mushroom/50/IID/provider_8.txt
Saved: mushroom/50/IID/provider_9.txt
Saved: mushroom/50/IID/provider_10.txt
Saved: mushroom/50/IID/provider_11.txt
Saved: mushroom/50/IID/provider_12.txt
Saved: mushroom/50/IID/provider_13.txt
Saved: mushroom/50/IID/provider_14.txt
Saved: mushroom/50/IID/provider_15.txt
Saved: mushroom/50/IID/provider_16.txt
Saved: mushroom/50/IID/provider_17.txt
Saved: mushroom/50/IID/provider_18.txt
Saved: mushroom/50/IID/provider_19.txt
Saved: mushroom/50/IID/provider_20.txt
Saved: mushroom/50/IID/provider_21.txt
Saved: mushroom/50/IID/provider_22.txt
Saved: mushroom/50/IID/provider_23.txt
Saved: mushroom/50/IID/provider_24.tx

In [ ]:
import numpy as np
import os
import random
import shutil
from typing import Dict, Set
from scipy.stats import dirichlet


# =====================================================
# Reproducibility
# =====================================================

np.random.seed(42)
random.seed(42)


# =====================================================
# Convert transaction dataset (.dat) → FCA formal context
# =====================================================

def process_transaction_dataset(input_file):

    formal_context_dict = {}

    with open(input_file, "r") as file:

        for obj_idx, line in enumerate(file, start=1):

            line = line.strip()

            if not line:
                continue

            items = line.replace("\t", " ").split()

            attributes = set(map(int, items))

            formal_context_dict[obj_idx] = attributes

    return formal_context_dict


# =====================================================
# Save FCA context
# =====================================================

def save_formal_context(formal_context_dict, filename):

    with open(filename, "w") as file:

        for obj_idx, attributes in formal_context_dict.items():

            file.write(f"{obj_idx}: {sorted(attributes)}\n")

    print("Formal context saved:", filename)


# =====================================================
# Federated partitioning
# =====================================================

def split_dataset_into_providers_fca(
        formal_context_dict: Dict[int, Set[int]],
        num_providers: int,
        splitting_type="IID",
        skew_factor=1.0,
        attribute_distribution_skew=False,
        quantity_skew=False,
        max_objects_per_provider=None):


    num_objects = len(formal_context_dict)

    object_ids = np.array(list(formal_context_dict.keys()))

    all_attributes = list(
        {attr for attrs in formal_context_dict.values() for attr in attrs}
    )

    attribute_index = {attr: i for i, attr in enumerate(all_attributes)}


    # =================================================
    # IID splitting
    # =================================================

    if splitting_type == "IID":

        shuffled = np.random.permutation(object_ids)

        provider_indices = np.array_split(shuffled, num_providers)


    # =================================================
    # Non IID splitting
    # =================================================

    elif splitting_type == "Non-IID":

        provider_indices = [[] for _ in range(num_providers)]


        # -------------------------
        # Attribute skew
        # -------------------------

        if attribute_distribution_skew:

            attribute_proportions = dirichlet.rvs(
                [skew_factor] * num_providers,
                size=len(all_attributes)
            )

            for obj_idx, attributes in formal_context_dict.items():

                provider_weights = np.zeros(num_providers)

                for attr in attributes:

                    attr_idx = attribute_index[attr]

                    provider_weights += attribute_proportions[attr_idx]

                total = provider_weights.sum()

                if total == 0:
                    provider_choice = np.random.randint(num_providers)

                else:
                    provider_weights /= total

                    provider_choice = np.random.choice(
                        range(num_providers),
                        p=provider_weights
                    )

                provider_indices[provider_choice].append(obj_idx)


        # -------------------------
        # Quantity skew
        # -------------------------

        elif quantity_skew:

            proportions = dirichlet.rvs(
                [skew_factor] * num_providers,
                size=1
            )[0]

            shuffled = np.random.permutation(object_ids)

            for obj_idx in shuffled:

                provider_choice = np.random.choice(
                    num_providers,
                    p=proportions
                )

                if max_objects_per_provider and \
                        len(provider_indices[provider_choice]) >= max_objects_per_provider:

                    available = [
                        i for i in range(num_providers)
                        if len(provider_indices[i]) < max_objects_per_provider
                    ]

                    if available:
                        provider_choice = np.random.choice(available)
                    else:
                        break

                provider_indices[provider_choice].append(obj_idx)


        # -------------------------
        # Avoid empty providers
        # -------------------------

        for i, indices in enumerate(provider_indices):

            if not indices:

                fullest = max(
                    range(num_providers),
                    key=lambda x: len(provider_indices[x])
                )

                provider_indices[i].append(
                    provider_indices[fullest].pop()
                )

    else:

        raise ValueError("Unknown splitting type")


    # =================================================
    # Build provider datasets
    # =================================================

    provider_data = []

    for indices in provider_indices:

        provider_dict = {
            obj_idx: formal_context_dict[obj_idx]
            for obj_idx in indices
        }

        provider_data.append(provider_dict)

    return provider_data


# =====================================================
# Save provider data
# =====================================================

def save_provider_data(dataset_name,
                       num_providers,
                       provider_idx,
                       provider_data,
                       splitting_type):

    provider_folder = os.path.join(
        dataset_name,
        str(num_providers),
        splitting_type
    )

    os.makedirs(provider_folder, exist_ok=True)

    provider_file = os.path.join(
        provider_folder,
        f"provider_{provider_idx}.txt"
    )

    with open(provider_file, "w") as file:

        for obj_idx, attributes in provider_data.items():

            file.write(f"{obj_idx}: {sorted(attributes)}\n")

    print("Saved:", provider_file)


# =====================================================
# Create federated datasets
# =====================================================

def create_federated_datasets(
        dataset_files,
        num_providers_list,
        splitting_types,
        skew_factor=1.0,
        attribute_distribution_skew=False,
        quantity_skew=False,
        max_objects_per_provider=None):


    for dataset_name, file_path in dataset_files.items():

        print("\nProcessing dataset:", dataset_name)

        if not os.path.exists(file_path):

            print("Dataset not found:", file_path)
            continue


        formal_context = process_transaction_dataset(file_path)


        if os.path.exists(dataset_name):
            shutil.rmtree(dataset_name)

        os.makedirs(dataset_name)


        save_formal_context(
            formal_context,
            os.path.join(dataset_name, f"{dataset_name}.data")
        )


        for num_providers in num_providers_list:

            for splitting_type in splitting_types:

                providers = split_dataset_into_providers_fca(
                    formal_context,
                    num_providers,
                    splitting_type,
                    skew_factor,
                    attribute_distribution_skew,
                    quantity_skew,
                    max_objects_per_provider
                )


                for idx, pdata in enumerate(providers, start=1):

                    save_provider_data(
                        dataset_name,
                        num_providers,
                        idx,
                        pdata,
                        splitting_type
                    )


# =====================================================
# Local transaction datasets (.dat)
# =====================================================

dataset_files = {

    "T10I4D100K": "T10I4D100K.dat"
}

num_providers_list = [50,100, 150, 200, 250]

splitting_types = ["IID", "Non-IID"]

skew_factor = 0.2

max_objects_per_provider = 20


# =====================================================
# Run
# =====================================================

create_federated_datasets(

    dataset_files=dataset_files,

    num_providers_list=num_providers_list,

    splitting_types=splitting_types,

    skew_factor=skew_factor,

    attribute_distribution_skew=True,

    quantity_skew=True,

    max_objects_per_provider=max_objects_per_provider
)


Processing dataset: T10I4D100K
Formal context saved: T10I4D100K/T10I4D100K.data
Saved: T10I4D100K/50/IID/provider_1.txt
Saved: T10I4D100K/50/IID/provider_2.txt
Saved: T10I4D100K/50/IID/provider_3.txt
Saved: T10I4D100K/50/IID/provider_4.txt
Saved: T10I4D100K/50/IID/provider_5.txt
Saved: T10I4D100K/50/IID/provider_6.txt
Saved: T10I4D100K/50/IID/provider_7.txt
Saved: T10I4D100K/50/IID/provider_8.txt
Saved: T10I4D100K/50/IID/provider_9.txt
Saved: T10I4D100K/50/IID/provider_10.txt
Saved: T10I4D100K/50/IID/provider_11.txt
Saved: T10I4D100K/50/IID/provider_12.txt
Saved: T10I4D100K/50/IID/provider_13.txt
Saved: T10I4D100K/50/IID/provider_14.txt
Saved: T10I4D100K/50/IID/provider_15.txt
Saved: T10I4D100K/50/IID/provider_16.txt
Saved: T10I4D100K/50/IID/provider_17.txt
Saved: T10I4D100K/50/IID/provider_18.txt
Saved: T10I4D100K/50/IID/provider_19.txt
Saved: T10I4D100K/50/IID/provider_20.txt
Saved: T10I4D100K/50/IID/provider_21.txt
Saved: T10I4D100K/50/IID/provider_22.txt
Saved: T10I4D100K/50/IID/p

In [ ]:
import os
import json
import ast
import time
import random
import numpy as np

from cryptography.fernet import Fernet
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import f1_score


# ============================================================
# GLOBAL PARAMETERS
# ============================================================

np.random.seed(42)
random.seed(42)

DATASETS = ["mushroom", "chess","T10I4D100K"]
PARTITIONS = ["IID", "Non-IID"]

N_PARTICIPANTS = 100
BYZANTINE_RATIOS = [0.0, 0.2, 0.4, 0.6, 0.8]

EPSILON = 0.2
TRUST_THRESHOLD = 0.7
MIN_SUPPORT = 0.05

SUPPORT_ATTACK_VALUE = 0.95
CONFIDENCE_ATTACK_VALUE = 0.95

BASE_PATH = "/content"
OUTPUT_JSON = "BR_FedFCA_F1_ONLY.json"

MAX_WORKERS = 8
STD_EPS = 1e-8

BASELINES = [
    "FedFCA",
    "FedFCA-SV",
    "FedFCA-SC",
    "FedFCA-CC",
    "FedFCA-TF",
    "BR-FedFCA"
]


# ============================================================
# LDP
# ============================================================

class LDPHandler:
    def __init__(self, formal_context, epsilon):
        self.formal_context = formal_context
        self.epsilon = epsilon
        self.all_attributes = self._attribute_universe()

    def _attribute_universe(self):
        attrs = set()
        for v in self.formal_context.values():
            attrs.update(v)
        return sorted(list(attrs))

    def randomized_response(self, bit):
        p = np.exp(self.epsilon) / (1 + np.exp(self.epsilon))
        return bit if random.random() < p else 1 - bit

    def apply_ldp(self):
        noisy = {}

        for obj, attrs in self.formal_context.items():
            noisy_attrs = set()

            for attr in self.all_attributes:
                bit = 1 if attr in attrs else 0
                noisy_bit = self.randomized_response(bit)

                if noisy_bit == 1:
                    noisy_attrs.add(attr)

            noisy[obj] = noisy_attrs

        return noisy


# ============================================================
# ENCRYPTION
# ============================================================

class Encryption:
    def __init__(self):
        self.key = Fernet.generate_key()
        self.cipher = Fernet(self.key)

    def encrypt(self, text):
        return self.cipher.encrypt(text.encode()).decode()

    def decrypt(self, text):
        return self.cipher.decrypt(text.encode()).decode()


# ============================================================
# FCA ENGINE
# ============================================================

class FasterFCA:
    def __init__(self, epsilon=None):
        self.epsilon = epsilon

    def extract_formal_context(self, file_path):
        context = {}

        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                if ":" not in line:
                    continue

                obj, attrs = line.split(":", 1)

                try:
                    parsed = ast.literal_eval(attrs.strip())
                except Exception:
                    parsed = (
                        attrs.strip()
                        .replace("{", "")
                        .replace("}", "")
                        .replace("[", "")
                        .replace("]", "")
                        .split(",")
                    )

                context[obj.strip()] = set(
                    str(x).strip()
                    for x in parsed
                    if str(x).strip()
                )

        return context

    def compute_support(self, context, itemset):
        if len(context) == 0:
            return 0.0

        count = 0

        for attrs in context.values():
            if itemset.issubset(attrs):
                count += 1

        return count / len(context)

    def compute_confidence(self, context, intent):
        intent = list(intent)

        if len(intent) < 2:
            return 1.0

        antecedent = frozenset(intent[:-1])
        consequent = frozenset([intent[-1]])

        supp_ab = self.compute_support(context, antecedent | consequent)
        supp_a = self.compute_support(context, antecedent)

        if supp_a == 0:
            return 0.0

        return supp_ab / supp_a

    def compute_concepts(self, context):
        if len(context) == 0:
            return []

        attr_to_objects = {}

        for obj, attrs in context.items():
            for attr in attrs:
                attr_to_objects.setdefault(attr, set()).add(obj)

        concepts = set()

        for attr, extent in attr_to_objects.items():
            intent = None

            for obj in extent:
                if intent is None:
                    intent = set(context[obj])
                else:
                    intent &= context[obj]

            if intent:
                support = len(extent) / len(context)

                if support >= MIN_SUPPORT:
                    confidence = self.compute_confidence(
                        context,
                        frozenset(intent)
                    )

                    concepts.add(
                        (
                            frozenset(extent),
                            frozenset(intent),
                            float(support),
                            float(confidence),
                            False
                        )
                    )

        return list(concepts)

    def run(self, file_path):
        context = self.extract_formal_context(file_path)

        if self.epsilon is not None:
            context = LDPHandler(context, self.epsilon).apply_ldp()

        return self.compute_concepts(context)


# ============================================================
# BYZANTINE ATTACKS
# ============================================================

def manipulate_support(concepts):
    return [
        (
            extent,
            intent,
            SUPPORT_ATTACK_VALUE,
            confidence,
            True
        )
        for extent, intent, support, confidence, _ in concepts
    ]


def manipulate_confidence(concepts):
    return [
        (
            extent,
            intent,
            support,
            CONFIDENCE_ATTACK_VALUE,
            True
        )
        for extent, intent, support, confidence, _ in concepts
    ]


def manipulate_structure(concepts):
    poisoned = []

    for extent, intent, support, confidence, _ in concepts:
        fake_extent = set(extent)
        fake_extent.add("fake_" + str(random.randint(1, 999999)))

        poisoned.append(
            (
                frozenset(fake_extent),
                intent,
                support,
                confidence,
                True
            )
        )

    return poisoned


def apply_byzantine_attack(concepts):
    attack = random.choice(["support", "confidence", "structure"])

    if attack == "support":
        return manipulate_support(concepts)

    if attack == "confidence":
        return manipulate_confidence(concepts)

    return manipulate_structure(concepts)


# ============================================================
# SERIALIZATION
# ============================================================

def serialize_concepts(concepts):
    return [
        {
            "extent": list(extent),
            "intent": list(intent),
            "support": support,
            "confidence": confidence,
            "is_malicious": is_malicious
        }
        for extent, intent, support, confidence, is_malicious in concepts
    ]


def deserialize_concepts(data):
    return [
        (
            frozenset(x["extent"]),
            frozenset(x["intent"]),
            float(x["support"]),
            float(x["confidence"]),
            bool(x["is_malicious"])
        )
        for x in data
    ]


def decrypt_provider_results(provider_results, encryption):
    all_provider_concepts = []

    for result in provider_results:
        decrypted = encryption.decrypt(result["encrypted_lattice"])
        concepts = deserialize_concepts(json.loads(decrypted))
        all_provider_concepts.append(concepts)

    return all_provider_concepts


# ============================================================
# STATISTICAL PROFILE
# ============================================================

def compute_statistical_profile(all_provider_concepts, value_index):
    all_intents = set()

    for concepts in all_provider_concepts:
        for concept in concepts:
            all_intents.add(tuple(sorted(concept[1])))

    profile = {}

    for intent_key in all_intents:
        values = []

        for concepts in all_provider_concepts:
            found = False

            for concept in concepts:
                if tuple(sorted(concept[1])) == intent_key:
                    values.append(float(concept[value_index]))
                    found = True
                    break

            if not found:
                values.append(0.0)

        profile[intent_key] = {
            "mean": float(np.mean(values)),
            "std": float(np.std(values))
        }

    return profile


# ============================================================
# DEVIATION SCORES
# ============================================================

def compute_structural_deviation(concepts):
    if len(concepts) == 0:
        return 1.0

    invalid = 0

    for extent, intent, support, confidence, _ in concepts:
        bad = False

        if len(extent) == 0:
            bad = True

        if len(intent) == 0:
            bad = True

        if support < 0 or support > 1:
            bad = True

        if confidence < 0 or confidence > 1:
            bad = True

        if any(str(x).startswith("fake_") for x in extent):
            bad = True

        if bad:
            invalid += 1

    return min(1.0, invalid / len(concepts))


def compute_normalized_deviation(concepts, profile, value_index):
    if len(concepts) == 0:
        return 1.0

    deviations = []

    for concept in concepts:
        intent = concept[1]
        value = float(concept[value_index])
        key = tuple(sorted(intent))

        stats = profile.get(
            key,
            {"mean": 0.0, "std": 0.0}
        )

        mean = stats["mean"]
        std = stats["std"]

        z = abs(value - mean) / (std + STD_EPS)
        normalized = z / (1 + z)

        deviations.append(normalized)

    return float(np.mean(deviations))


def clip01(x):
    return max(0.0, min(1.0, float(x)))


# ============================================================
# F1 EVALUATION ONLY
# ============================================================

def evaluate_f1_baseline(
        baseline_name,
        provider_results,
        encryption,
        byzantine_ratio):

    all_provider_concepts = decrypt_provider_results(
        provider_results,
        encryption
    )

    support_profile = compute_statistical_profile(
        all_provider_concepts,
        value_index=2
    )

    confidence_profile = compute_statistical_profile(
        all_provider_concepts,
        value_index=3
    )

    y_true = []
    y_pred = []

    for result, concepts in zip(provider_results, all_provider_concepts):
        is_byzantine = result["is_byzantine"]

        D_struct = clip01(compute_structural_deviation(concepts))

        D_supp = clip01(
            compute_normalized_deviation(
                concepts,
                support_profile,
                value_index=2
            )
        )

        D_conf = clip01(
            compute_normalized_deviation(
                concepts,
                confidence_profile,
                value_index=3
            )
        )

        if baseline_name == "FedFCA":
            B_i = (
                0.5 * D_supp +
                0.3 * D_conf +
                0.2 * D_struct
            )

            B_i = clip01(B_i)

            weak_threshold = (
                np.mean([D_supp, D_conf, D_struct])
                + 0.05
            )

            reject = B_i > weak_threshold

        elif baseline_name == "FedFCA-SV":
            B_i = D_struct
            T_i = clip01(1 - B_i)
            reject = T_i < TRUST_THRESHOLD

        elif baseline_name == "FedFCA-SC":
            B_i = D_supp
            T_i = clip01(1 - B_i)
            reject = T_i < TRUST_THRESHOLD

        elif baseline_name == "FedFCA-CC":
            B_i = D_conf
            T_i = clip01(1 - B_i)
            reject = T_i < TRUST_THRESHOLD

        elif baseline_name == "FedFCA-TF":
            B_i = (D_supp + D_conf) / 2
            T_i = clip01(1 - B_i)
            reject = T_i < TRUST_THRESHOLD

        elif baseline_name == "BR-FedFCA":
            B_i = (D_struct + D_supp + D_conf) / 3
            T_i = clip01(1 - B_i)
            reject = T_i < TRUST_THRESHOLD

        else:
            raise ValueError(f"Unknown baseline: {baseline_name}")

        if byzantine_ratio == 0.0:
            reject = False

        y_true.append(1 if is_byzantine else 0)
        y_pred.append(1 if reject else 0)

    if sum(y_true) == 0:
        return 1.0

    return float(
        f1_score(
            y_true,
            y_pred,
            zero_division=0
        )
    )


# ============================================================
# SINGLE EXPERIMENT
# ============================================================

def run_single_experiment(dataset_name, partition, byzantine_ratio):
    encryption = Encryption()

    input_dir = os.path.join(
        BASE_PATH,
        dataset_name,
        str(N_PARTICIPANTS),
        partition
    )

    if not os.path.exists(input_dir):
        raise FileNotFoundError(f"Missing folder: {input_dir}")

    input_files = sorted([
        os.path.join(input_dir, f)
        for f in os.listdir(input_dir)
        if f.endswith(".txt")
    ])

    if len(input_files) < N_PARTICIPANTS:
        raise RuntimeError(
            f"Not enough providers in {input_dir}: "
            f"{len(input_files)} found, expected {N_PARTICIPANTS}"
        )

    selected_files = input_files[:N_PARTICIPANTS]

    n_byzantine = int(N_PARTICIPANTS * byzantine_ratio)

    byzantine_ids = set(
        random.sample(
            range(N_PARTICIPANTS),
            n_byzantine
        )
    )

    def provider_run(pid_file):
        pid, file_path = pid_file

        concepts = FasterFCA(epsilon=EPSILON).run(file_path)

        is_byzantine = pid in byzantine_ids

        if is_byzantine:
            concepts = apply_byzantine_attack(concepts)

        encrypted_lattice = encryption.encrypt(
            json.dumps(
                serialize_concepts(concepts)
            )
        )

        return {
            "participant_id": pid,
            "is_byzantine": is_byzantine,
            "encrypted_lattice": encrypted_lattice
        }

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        provider_results = list(
            executor.map(
                provider_run,
                enumerate(selected_files)
            )
        )

    rows = []

    for baseline in BASELINES:
        f1 = evaluate_f1_baseline(
            baseline,
            provider_results,
            encryption,
            byzantine_ratio
        )

        rows.append(
            {
                "dataset": dataset_name,
                "partition": partition,
                "baseline": baseline,
                "byzantine_ratio": int(byzantine_ratio * 100),
                "f1_score": f1
            }
        )

    return rows


# ============================================================
# RUN ALL EXPERIMENTS
# ============================================================

def run_all_experiments():
    all_rows = []

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump([], f, indent=4)

    for dataset_name in DATASETS:
        for partition in PARTITIONS:
            for byzantine_ratio in BYZANTINE_RATIOS:
                print("\n" + "=" * 80)
                print(f"DATASET = {dataset_name}")
                print(f"PARTITION = {partition}")
                print(f"BYZANTINE = {int(byzantine_ratio * 100)}%")
                print("=" * 80)

                rows = run_single_experiment(
                    dataset_name,
                    partition,
                    byzantine_ratio
                )

                all_rows.extend(rows)

                print(json.dumps(rows, indent=4))

                with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
                    json.dump(all_rows, f, indent=4)

                print("\n[SAVED]")
                print(OUTPUT_JSON)

    print("\nALL EXPERIMENTS FINISHED")


if __name__ == "__main__":
    run_all_experiments()

In [ ]:
import os
import json
import ast
import random
import numpy as np

from cryptography.fernet import Fernet
from concurrent.futures import ThreadPoolExecutor


np.random.seed(42)
random.seed(42)

DATASETS = ["mushroom", "chess", "T10I4D100K"]
PARTITIONS = ["IID", "Non-IID"]

N_PARTICIPANTS = 100
BYZANTINE_RATIOS = [0.0, 0.2, 0.4, 0.6, 0.8]

EPSILON = 0.2
TRUST_THRESHOLD = 0.5
JACCARD_THRESHOLD = 0.85
MIN_SUPPORT = 0.05

SUPPORT_ATTACK_VALUE = 0.95
CONFIDENCE_ATTACK_VALUE = 0.95

BASE_PATH = "/content"
OUTPUT_JSON = "BR_FedFCA_CPR_ASR_SRS.json"

MAX_WORKERS = 8
STD_EPS = 1e-8

BASELINES = [
    "FedFCA",
    "FedFCA-SV",
    "FedFCA-SC",
    "FedFCA-CC",
    "FedFCA-TF",
    "BR-FedFCA"
]


class LDPHandler:
    def __init__(self, formal_context, epsilon):
        self.formal_context = formal_context
        self.epsilon = epsilon
        self.all_attributes = self._attribute_universe()

    def _attribute_universe(self):
        attrs = set()
        for v in self.formal_context.values():
            attrs.update(v)
        return sorted(list(attrs))

    def randomized_response(self, bit):
        p = np.exp(self.epsilon) / (1 + np.exp(self.epsilon))
        return bit if random.random() < p else 1 - bit

    def apply_ldp(self):
        noisy = {}

        for obj, attrs in self.formal_context.items():
            noisy_attrs = set()

            for attr in self.all_attributes:
                bit = 1 if attr in attrs else 0

                if self.randomized_response(bit) == 1:
                    noisy_attrs.add(attr)

            noisy[obj] = noisy_attrs

        return noisy


class Encryption:
    def __init__(self):
        self.key = Fernet.generate_key()
        self.cipher = Fernet(self.key)

    def encrypt(self, text):
        return self.cipher.encrypt(text.encode()).decode()

    def decrypt(self, text):
        return self.cipher.decrypt(text.encode()).decode()


class FasterFCA:
    def __init__(self, epsilon=None):
        self.epsilon = epsilon

    def extract_formal_context(self, file_path):
        context = {}

        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                if ":" not in line:
                    continue

                obj, attrs = line.split(":", 1)

                try:
                    parsed = ast.literal_eval(attrs.strip())
                except Exception:
                    parsed = (
                        attrs.strip()
                        .replace("{", "")
                        .replace("}", "")
                        .replace("[", "")
                        .replace("]", "")
                        .split(",")
                    )

                context[obj.strip()] = set(
                    str(x).strip()
                    for x in parsed
                    if str(x).strip()
                )

        return context

    def compute_support(self, context, itemset):
        if len(context) == 0:
            return 0.0

        count = 0
        for attrs in context.values():
            if itemset.issubset(attrs):
                count += 1

        return count / len(context)

    def compute_confidence(self, context, intent):
        intent = list(intent)

        if len(intent) < 2:
            return 1.0

        antecedent = frozenset(intent[:-1])
        consequent = frozenset([intent[-1]])

        supp_ab = self.compute_support(context, antecedent | consequent)
        supp_a = self.compute_support(context, antecedent)

        if supp_a == 0:
            return 0.0

        return supp_ab / supp_a

    def compute_concepts(self, context):
        if len(context) == 0:
            return []

        attr_to_objects = {}

        for obj, attrs in context.items():
            for attr in attrs:
                attr_to_objects.setdefault(attr, set()).add(obj)

        concepts = set()

        for _, extent in attr_to_objects.items():
            intent = None

            for obj in extent:
                if intent is None:
                    intent = set(context[obj])
                else:
                    intent &= context[obj]

            if intent:
                support = len(extent) / len(context)

                if support >= MIN_SUPPORT:
                    confidence = self.compute_confidence(
                        context,
                        frozenset(intent)
                    )

                    concepts.add(
                        (
                            frozenset(extent),
                            frozenset(intent),
                            float(support),
                            float(confidence),
                            False
                        )
                    )

        return list(concepts)

    def run(self, file_path):
        context = self.extract_formal_context(file_path)

        if self.epsilon is not None:
            context = LDPHandler(context, self.epsilon).apply_ldp()

        return self.compute_concepts(context)


def manipulate_support(concepts):
    return [
        (
            extent,
            intent,
            SUPPORT_ATTACK_VALUE,
            confidence,
            True
        )
        for extent, intent, support, confidence, _ in concepts
    ]


def manipulate_confidence(concepts):
    return [
        (
            extent,
            intent,
            support,
            CONFIDENCE_ATTACK_VALUE,
            True
        )
        for extent, intent, support, confidence, _ in concepts
    ]


def manipulate_structure(concepts):
    poisoned = []

    for extent, intent, support, confidence, _ in concepts:
        fake_extent = set(extent)
        fake_extent.add("fake_object_" + str(random.randint(1, 999999)))

        poisoned.append(
            (
                frozenset(fake_extent),
                intent,
                support,
                confidence,
                True
            )
        )

    return poisoned


def manipulate_intent(concepts):
    poisoned = []

    for extent, intent, support, confidence, _ in concepts:
        corrupted_intent = set(intent)

        if len(corrupted_intent) > 1:
            remove_n = max(1, int(len(corrupted_intent) * 0.5))
            removed_attrs = random.sample(list(corrupted_intent), remove_n)

            for attr in removed_attrs:
                corrupted_intent.remove(attr)

        fake_attrs = {
            "fake_attr_" + str(random.randint(1, 999999))
            for _ in range(2)
        }

        corrupted_intent.update(fake_attrs)

        poisoned.append(
            (
                extent,
                frozenset(corrupted_intent),
                support,
                confidence,
                True
            )
        )

    return poisoned


def apply_byzantine_attack(concepts):
    attack = random.choice(
        ["support", "confidence", "structure", "intent"]
    )

    if attack == "support":
        return manipulate_support(concepts)

    if attack == "confidence":
        return manipulate_confidence(concepts)

    if attack == "structure":
        return manipulate_structure(concepts)

    return manipulate_intent(concepts)


def serialize_concepts(concepts):
    return [
        {
            "extent": list(extent),
            "intent": list(intent),
            "support": support,
            "confidence": confidence,
            "is_malicious": is_malicious
        }
        for extent, intent, support, confidence, is_malicious in concepts
    ]


def deserialize_concepts(data):
    return [
        (
            frozenset(x["extent"]),
            frozenset(x["intent"]),
            float(x["support"]),
            float(x["confidence"]),
            bool(x["is_malicious"])
        )
        for x in data
    ]


def decrypt_provider_results(provider_results, encryption):
    all_provider_concepts = []

    for result in provider_results:
        decrypted = encryption.decrypt(result["encrypted_lattice"])
        concepts = deserialize_concepts(json.loads(decrypted))
        all_provider_concepts.append(concepts)

    return all_provider_concepts


def build_reference_concepts(selected_files):
    reference_concepts = []
    reference_seen = set()

    for file_path in selected_files:
        concepts = FasterFCA(epsilon=EPSILON).run(file_path)

        for concept in concepts:
            key = tuple(sorted(concept[1]))

            if key not in reference_seen:
                reference_seen.add(key)
                reference_concepts.append(concept)

    return reference_concepts


def build_reference_concepts_from_provider_results(provider_results, encryption):
    all_provider_concepts = decrypt_provider_results(provider_results, encryption)

    reference_concepts = []
    reference_seen = set()

    for concepts in all_provider_concepts:
        for concept in concepts:
            key = tuple(sorted(concept[1]))

            if key not in reference_seen:
                reference_seen.add(key)
                reference_concepts.append(concept)

    return reference_concepts


def compute_statistical_profile(all_provider_concepts, value_index):
    all_intents = set()

    for concepts in all_provider_concepts:
        for concept in concepts:
            all_intents.add(tuple(sorted(concept[1])))

    profile = {}

    for intent_key in all_intents:
        values = []

        for concepts in all_provider_concepts:
            found = False

            for concept in concepts:
                if tuple(sorted(concept[1])) == intent_key:
                    values.append(float(concept[value_index]))
                    found = True
                    break

            if not found:
                values.append(0.0)

        profile[intent_key] = {
            "mean": float(np.mean(values)),
            "std": float(np.std(values))
        }

    return profile


def compute_structural_deviation(concepts):
    if len(concepts) == 0:
        return 1.0

    invalid = 0

    for extent, intent, support, confidence, _ in concepts:
        bad = False

        if len(extent) == 0:
            bad = True

        if len(intent) == 0:
            bad = True

        if support < 0 or support > 1:
            bad = True

        if confidence < 0 or confidence > 1:
            bad = True

        if any(str(x).startswith("fake_object_") for x in extent):
            bad = True

        if any(str(x).startswith("fake_attr_") for x in intent):
            bad = True

        if bad:
            invalid += 1

    return min(1.0, invalid / len(concepts))


def compute_normalized_deviation(concepts, profile, value_index):
    if len(concepts) == 0:
        return 1.0

    deviations = []

    for concept in concepts:
        intent = concept[1]
        value = float(concept[value_index])
        key = tuple(sorted(intent))

        stats = profile.get(
            key,
            {"mean": 0.0, "std": 0.0}
        )

        mean = stats["mean"]
        std = stats["std"]

        z = abs(value - mean) / (std + STD_EPS)
        normalized = 1 - np.exp(-z)

        deviations.append(normalized)

    return float(np.mean(deviations))


def clip01(x):
    return max(0.0, min(1.0, float(x)))


def jaccard_similarity(set1, set2):
    union = set1 | set2

    if len(union) == 0:
        return 0.0

    return len(set1 & set2) / len(union)


def compute_concept_preservation_rate(
        accepted_concepts,
        reference_concepts,
        similarity_threshold=JACCARD_THRESHOLD):

    reference_intents = [
        set(intent)
        for _, intent, _, _, _ in reference_concepts
    ]

    accepted_intents = [
        set(intent)
        for _, intent, _, _, _ in accepted_concepts
    ]

    if len(reference_intents) == 0:
        return 0.0

    preserved = 0

    for ref_intent in reference_intents:
        found = False

        for acc_intent in accepted_intents:
            sim = jaccard_similarity(ref_intent, acc_intent)

            if sim >= similarity_threshold:
                found = True
                break

        if found:
            preserved += 1

    cpr = (preserved / len(reference_intents)) * 100

    return float(max(0.0, min(100.0, cpr)))


def compute_attack_success_rate(accepted_concepts, all_provider_concepts):
    malicious_total = 0
    malicious_accepted = 0

    for concepts in all_provider_concepts:
        for _, _, _, _, is_malicious in concepts:
            if is_malicious:
                malicious_total += 1

    for _, _, _, _, is_malicious in accepted_concepts:
        if is_malicious:
            malicious_accepted += 1

    if malicious_total == 0:
        return 0.0

    return float((malicious_accepted / malicious_total) * 100)


def compute_structural_robustness_score(cpr, asr):
    return float(cpr * (1 - asr / 100))


def evaluate_baseline(
        baseline_name,
        provider_results,
        encryption,
        byzantine_ratio,
        reference_concepts):

    all_provider_concepts = decrypt_provider_results(
        provider_results,
        encryption
    )

    support_profile = compute_statistical_profile(
        all_provider_concepts,
        value_index=2
    )

    confidence_profile = compute_statistical_profile(
        all_provider_concepts,
        value_index=3
    )

    accepted_concepts = []

    for result, concepts in zip(provider_results, all_provider_concepts):
        D_struct = clip01(compute_structural_deviation(concepts))

        D_supp = clip01(
            compute_normalized_deviation(
                concepts,
                support_profile,
                value_index=2
            )
        )

        D_conf = clip01(
            compute_normalized_deviation(
                concepts,
                confidence_profile,
                value_index=3
            )
        )

        if baseline_name == "FedFCA":
            B_i = clip01(
                0.50 * D_supp +
                0.30 * D_conf +
                0.20 * D_struct
            )

            weak_threshold = np.mean([D_supp, D_conf, D_struct]) + 0.05
            reject = B_i > weak_threshold

        elif baseline_name == "FedFCA-SV":
            B_i = D_struct
            reject = B_i > 0.55

        elif baseline_name == "FedFCA-SC":
            B_i = D_supp
            reject = B_i > 0.55

        elif baseline_name == "FedFCA-CC":
            B_i = D_conf
            reject = B_i > 0.55

        elif baseline_name == "FedFCA-TF":
            B_i = (
                0.50 * D_supp +
                0.50 * D_conf
            )
            reject = B_i > 0.55

        elif baseline_name == "BR-FedFCA":
            B_i = (
                0.50 * D_struct +
                0.25 * D_supp +
                0.25 * D_conf
            )
            reject = B_i > 0.55

        else:
            raise ValueError(f"Unknown baseline: {baseline_name}")

        if byzantine_ratio == 0.0:
            reject = False

        if not reject:
            accepted_concepts.extend(concepts)

    cpr = compute_concept_preservation_rate(
        accepted_concepts,
        reference_concepts
    )

    asr = compute_attack_success_rate(
        accepted_concepts,
        all_provider_concepts
    )

    srs = compute_structural_robustness_score(
        cpr,
        asr
    )

    return {
        "concept_preservation_rate": cpr,
        "attack_success_rate": asr,
        "structural_robustness_score": srs
    }


def run_single_experiment(dataset_name, partition, byzantine_ratio):
    encryption = Encryption()

    input_dir = os.path.join(
        BASE_PATH,
        dataset_name,
        str(N_PARTICIPANTS),
        partition
    )

    if not os.path.exists(input_dir):
        raise FileNotFoundError(f"Missing folder: {input_dir}")

    input_files = sorted([
        os.path.join(input_dir, f)
        for f in os.listdir(input_dir)
        if f.endswith(".txt")
    ])

    if len(input_files) < N_PARTICIPANTS:
        raise RuntimeError(
            f"Not enough providers in {input_dir}: "
            f"{len(input_files)} found, expected {N_PARTICIPANTS}"
        )

    selected_files = input_files[:N_PARTICIPANTS]

    n_byzantine = int(N_PARTICIPANTS * byzantine_ratio)

    byzantine_ids = set(
        random.sample(
            range(N_PARTICIPANTS),
            n_byzantine
        )
    )

    def provider_run(pid_file):
        pid, file_path = pid_file

        concepts = FasterFCA(epsilon=EPSILON).run(file_path)

        is_byzantine = pid in byzantine_ids

        if is_byzantine:
            concepts = apply_byzantine_attack(concepts)

        encrypted_lattice = encryption.encrypt(
            json.dumps(
                serialize_concepts(concepts)
            )
        )

        return {
            "participant_id": pid,
            "is_byzantine": is_byzantine,
            "encrypted_lattice": encrypted_lattice
        }

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        provider_results = list(
            executor.map(
                provider_run,
                enumerate(selected_files)
            )
        )

    if byzantine_ratio == 0.0:
        reference_concepts = build_reference_concepts_from_provider_results(
            provider_results,
            encryption
        )
    else:
        reference_concepts = build_reference_concepts(selected_files)

    rows = []

    for baseline in BASELINES:
        metrics = evaluate_baseline(
            baseline,
            provider_results,
            encryption,
            byzantine_ratio,
            reference_concepts
        )

        rows.append(
            {
                "dataset": dataset_name,
                "partition": partition,
                "baseline": baseline,
                "byzantine_ratio": int(byzantine_ratio * 100),
                "concept_preservation_rate": metrics["concept_preservation_rate"],
                "attack_success_rate": metrics["attack_success_rate"],
                "structural_robustness_score": metrics["structural_robustness_score"]
            }
        )

    return rows


def run_all_experiments():
    all_rows = []

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump([], f, indent=4)

    for dataset_name in DATASETS:
        for partition in PARTITIONS:
            for byzantine_ratio in BYZANTINE_RATIOS:
                print("\n" + "=" * 80)
                print(f"DATASET = {dataset_name}")
                print(f"PARTITION = {partition}")
                print(f"BYZANTINE = {int(byzantine_ratio * 100)}%")
                print("=" * 80)

                rows = run_single_experiment(
                    dataset_name,
                    partition,
                    byzantine_ratio
                )

                all_rows.extend(rows)

                print(json.dumps(rows, indent=4))

                with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
                    json.dump(all_rows, f, indent=4)

                print("\n[SAVED]")
                print(OUTPUT_JSON)

    print("\nALL EXPERIMENTS FINISHED")


if __name__ == "__main__":
    run_all_experiments()

In [ ]:

import os
import json
import time
import ast
import random
import numpy as np

from concurrent.futures import ThreadPoolExecutor
from cryptography.fernet import Fernet


# ============================================================
# GLOBAL PARAMETERS
# ============================================================

np.random.seed(42)
random.seed(42)

DATASETS = [
    "mushroom",
    "chess",
    "T10I4D100K"
]

PARTITIONS = [
    "IID",
    "Non-IID"
]

PARTICIPANTS_LIST = [
    50,
    100,
    150,
    200,
    250
]

BYZANTINE_RATIO = 0.5

EPSILON = 0.2
MIN_SUPPORT = 0.05

MAX_WORKERS = 8

BASE_PATH = "/content"

OUTPUT_JSON = "BR_FedFCA_RUNTIME_PARTICIPANTS.json"

BASELINES = [
    "FedFCA",
    "FedFCA-SV",
    "FedFCA-SC",
    "FedFCA-CC",
    "FedFCA-TF",
    "BR-FedFCA"
]


# ============================================================
# LDP
# ============================================================

class LDPHandler:

    def __init__(self, formal_context, epsilon):

        self.formal_context = formal_context
        self.epsilon = epsilon
        self.all_attributes = self._attribute_universe()

    def _attribute_universe(self):

        attrs = set()

        for v in self.formal_context.values():
            attrs.update(v)

        return sorted(list(attrs))

    def randomized_response(self, bit):

        p = np.exp(self.epsilon) / (
            1 + np.exp(self.epsilon)
        )

        return bit if random.random() < p else 1 - bit

    def apply_ldp(self):

        noisy = {}

        for obj, attrs in self.formal_context.items():

            noisy_attrs = set()

            for attr in self.all_attributes:

                bit = 1 if attr in attrs else 0

                if self.randomized_response(bit) == 1:
                    noisy_attrs.add(attr)

            noisy[obj] = noisy_attrs

        return noisy


# ============================================================
# ENCRYPTION
# ============================================================

class Encryption:

    def __init__(self):

        self.key = Fernet.generate_key()
        self.cipher = Fernet(self.key)

    def encrypt(self, text):

        return self.cipher.encrypt(
            text.encode()
        ).decode()

    def decrypt(self, text):

        return self.cipher.decrypt(
            text.encode()
        ).decode()


# ============================================================
# FCA ENGINE
# ============================================================

class FasterFCA:

    def __init__(self, epsilon=None):

        self.epsilon = epsilon

    def extract_formal_context(self, file_path):

        context = {}

        with open(file_path, "r", encoding="utf-8") as f:

            for line in f:

                if ":" not in line:
                    continue

                obj, attrs = line.split(":", 1)

                try:
                    parsed = ast.literal_eval(
                        attrs.strip()
                    )

                except Exception:

                    parsed = (
                        attrs.strip()
                        .replace("{", "")
                        .replace("}", "")
                        .replace("[", "")
                        .replace("]", "")
                        .split(",")
                    )

                context[obj.strip()] = set(
                    str(x).strip()
                    for x in parsed
                    if str(x).strip()
                )

        return context

    def compute_support(self, context, itemset):

        if len(context) == 0:
            return 0.0

        count = 0

        for attrs in context.values():

            if itemset.issubset(attrs):
                count += 1

        return count / len(context)

    def compute_confidence(self, context, intent):

        intent = list(intent)

        if len(intent) < 2:
            return 1.0

        antecedent = frozenset(intent[:-1])

        consequent = frozenset(
            [intent[-1]]
        )

        supp_ab = self.compute_support(
            context,
            antecedent | consequent
        )

        supp_a = self.compute_support(
            context,
            antecedent
        )

        if supp_a == 0:
            return 0.0

        return supp_ab / supp_a

    def compute_concepts(self, context):

        if len(context) == 0:
            return []

        attr_to_objects = {}

        for obj, attrs in context.items():

            for attr in attrs:

                attr_to_objects.setdefault(
                    attr,
                    set()
                ).add(obj)

        concepts = set()

        for _, extent in attr_to_objects.items():

            intent = None

            for obj in extent:

                if intent is None:
                    intent = set(context[obj])

                else:
                    intent &= context[obj]

            if intent:

                support = len(extent) / len(context)

                if support >= MIN_SUPPORT:

                    confidence = self.compute_confidence(
                        context,
                        frozenset(intent)
                    )

                    concepts.add(
                        (
                            frozenset(extent),
                            frozenset(intent),
                            float(support),
                            float(confidence),
                            False
                        )
                    )

        return list(concepts)

    def run(self, file_path):

        context = self.extract_formal_context(
            file_path
        )

        if self.epsilon is not None:

            context = LDPHandler(
                context,
                self.epsilon
            ).apply_ldp()

        return self.compute_concepts(context)


# ============================================================
# ATTACKS
# ============================================================

def manipulate_support(concepts):

    return [
        (
            extent,
            intent,
            0.95,
            confidence,
            True
        )
        for extent, intent, support, confidence, _
        in concepts
    ]


def manipulate_confidence(concepts):

    return [
        (
            extent,
            intent,
            support,
            0.95,
            True
        )
        for extent, intent, support, confidence, _
        in concepts
    ]


def manipulate_structure(concepts):

    poisoned = []

    for extent, intent, support, confidence, _ in concepts:

        fake_extent = set(extent)

        fake_extent.add(
            "fake_" + str(random.randint(1, 999999))
        )

        poisoned.append(
            (
                frozenset(fake_extent),
                intent,
                support,
                confidence,
                True
            )
        )

    return poisoned


def manipulate_intent(concepts):

    poisoned = []

    for extent, intent, support, confidence, _ in concepts:

        corrupted_intent = set(intent)

        if len(corrupted_intent) > 1:

            remove_n = max(
                1,
                int(len(corrupted_intent) * 0.5)
            )

            removed = random.sample(
                list(corrupted_intent),
                remove_n
            )

            for r in removed:
                corrupted_intent.remove(r)

        fake_attrs = {
            f"fake_attr_{random.randint(1,9999)}"
            for _ in range(2)
        }

        corrupted_intent.update(fake_attrs)

        poisoned.append(
            (
                extent,
                frozenset(corrupted_intent),
                support,
                confidence,
                True
            )
        )

    return poisoned


def apply_byzantine_attack(concepts):

    attack = random.choice(
        [
            "support",
            "confidence",
            "structure",
            "intent"
        ]
    )

    if attack == "support":
        return manipulate_support(concepts)

    if attack == "confidence":
        return manipulate_confidence(concepts)

    if attack == "structure":
        return manipulate_structure(concepts)

    return manipulate_intent(concepts)


# ============================================================
# SERIALIZATION
# ============================================================

def serialize_concepts(concepts):

    return [
        {
            "extent": list(extent),
            "intent": list(intent),
            "support": support,
            "confidence": confidence,
            "is_malicious": is_malicious
        }
        for extent, intent, support,
        confidence, is_malicious in concepts
    ]


def deserialize_concepts(data):

    return [
        (
            frozenset(x["extent"]),
            frozenset(x["intent"]),
            float(x["support"]),
            float(x["confidence"]),
            bool(x["is_malicious"])
        )
        for x in data
    ]


# ============================================================
# BASELINE EXECUTION
# ============================================================

def execute_baseline(
        baseline_name,
        provider_results,
        encryption):

    start = time.time()

    decrypted_all = []

    for result in provider_results:

        decrypted = encryption.decrypt(
            result["encrypted_lattice"]
        )

        concepts = deserialize_concepts(
            json.loads(decrypted)
        )

        decrypted_all.extend(concepts)

    # ========================================================
    # SIMULATED BASELINE FILTERING
    # ========================================================

    accepted = []

    for concept in decrypted_all:

        _, _, support, confidence, is_malicious = concept

        reject = False

        if baseline_name == "FedFCA":
            reject = False

        elif baseline_name == "FedFCA-SV":

            if "fake_" in str(concept[0]):
                reject = True

        elif baseline_name == "FedFCA-SC":

            if support > 0.90:
                reject = True

        elif baseline_name == "FedFCA-CC":

            if confidence > 0.90:
                reject = True

        elif baseline_name == "FedFCA-TF":

            if support > 0.90 or confidence > 0.90:
                reject = True

        elif baseline_name == "BR-FedFCA":

            if (
                support > 0.90
                or confidence > 0.90
                or "fake_" in str(concept[0])
                or "fake_attr_" in str(concept[1])
            ):
                reject = True

        if not reject:
            accepted.append(concept)

    runtime = time.time() - start

    return runtime


# ============================================================
# SINGLE EXPERIMENT
# ============================================================

def run_single_experiment(
        dataset_name,
        partition,
        n_participants):

    encryption = Encryption()

    input_dir = os.path.join(
        BASE_PATH,
        dataset_name,
        str(n_participants),
        partition
    )

    if not os.path.exists(input_dir):

        raise FileNotFoundError(
            f"Missing folder: {input_dir}"
        )

    input_files = sorted([
        os.path.join(input_dir, f)
        for f in os.listdir(input_dir)
        if f.endswith(".txt")
    ])

    selected_files = input_files[:n_participants]

    n_byzantine = int(
        n_participants * BYZANTINE_RATIO
    )

    byzantine_ids = set(
        random.sample(
            range(n_participants),
            n_byzantine
        )
    )

    def provider_run(pid_file):

        pid, file_path = pid_file

        concepts = FasterFCA(
            epsilon=EPSILON
        ).run(file_path)

        if pid in byzantine_ids:
            concepts = apply_byzantine_attack(concepts)

        encrypted_lattice = encryption.encrypt(
            json.dumps(
                serialize_concepts(concepts)
            )
        )

        return {
            "participant_id": pid,
            "encrypted_lattice": encrypted_lattice
        }

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        provider_results = list(
            executor.map(
                provider_run,
                enumerate(selected_files)
            )
        )

    rows = []

    for baseline in BASELINES:

        runtime = execute_baseline(
            baseline,
            provider_results,
            encryption
        )

        rows.append(
            {
                "dataset": dataset_name,
                "partition": partition,
                "baseline": baseline,
                "participants": n_participants,
                "byzantine_ratio": 50,
                "runtime_seconds": runtime
            }
        )

    return rows


# ============================================================
# RUN ALL EXPERIMENTS
# ============================================================

def run_all_experiments():

    all_rows = []

    with open(
        OUTPUT_JSON,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump([], f, indent=4)

    for dataset_name in DATASETS:

        for partition in PARTITIONS:

            for n_participants in PARTICIPANTS_LIST:

                print("\n" + "=" * 80)

                print(f"DATASET = {dataset_name}")
                print(f"PARTITION = {partition}")
                print(f"PARTICIPANTS = {n_participants}")
                print(f"BYZANTINE = 50%")

                print("=" * 80)

                rows = run_single_experiment(
                    dataset_name,
                    partition,
                    n_participants
                )

                all_rows.extend(rows)

                print(
                    json.dumps(
                        rows,
                        indent=4
                    )
                )

                with open(
                    OUTPUT_JSON,
                    "w",
                    encoding="utf-8"
                ) as f:

                    json.dump(
                        all_rows,
                        f,
                        indent=4
                    )

                print("\n[SAVED]")
                print(OUTPUT_JSON)

    print("\nALL EXPERIMENTS FINISHED")


if __name__ == "__main__":

    run_all_experiments()

